# Credit Risk Demo - Data Preparation

This notebook prepares the **Kaggle Credit Risk dataset** for ensemble model training.

**Ensemble Strategy**: All models use all features. Diversity comes from different algorithms analyzing the same data in different ways.

## Steps
1. Install dependencies
2. Download dataset from Kaggle
3. Load and explore data
4. Handle missing values and encode categorical features
5. Create feature set (all 11 features for all models)
6. Train/Test split
7. Save processed data

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install -q pandas numpy scikit-learn matplotlib seaborn kaggle

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import os

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

print("✓ Libraries imported successfully")

## 2. Download Dataset from Kaggle

**Note:** This dataset is public and does not require Kaggle API authentication.

In [ ]:
# Download dataset from Kaggle (public dataset, no authentication required)
os.makedirs('data', exist_ok=True)
!kaggle datasets download -d laotse/credit-risk-dataset -p data --unzip

print("✓ Dataset downloaded to data/")

## 3. Load and Explore Data

In [ ]:
# Load dataset
df = pd.read_csv('data/credit_risk_dataset.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Dataset info
print("Dataset Info:")
print("="*60)
df.info()

print("\n" + "="*60)
print("Statistical Summary:")
print("="*60)
df.describe()

In [ ]:
# Identify target variable
target_col = 'loan_status'

print(f"Target variable: {target_col}")
print(f"\nClass distribution:")
print(df[target_col].value_counts())
print(f"\nClass proportions:")
print(df[target_col].value_counts(normalize=True))

## 4. Handle Missing Values and Encoding

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_df) > 0:
    print("Missing Values:")
    print(missing_df)
else:
    print("✓ No missing values found")

In [ ]:
# Handle missing values
df_clean = df.copy()

# CRITICAL: Drop rows with missing target variable (can't train without labels)
if df_clean[target_col].isnull().sum() > 0:
    rows_before = len(df_clean)
    df_clean = df_clean.dropna(subset=[target_col])
    rows_dropped = rows_before - len(df_clean)
    print(f"⚠ Dropped {rows_dropped} rows with missing target variable")
    print(f"Remaining rows: {len(df_clean)}")

# Identify numerical and categorical columns (excluding target)
numerical_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
numerical_cols = [col for col in numerical_cols if col != target_col]

categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [col for col in categorical_cols if col != target_col]

# Fill numerical missing values with median
for col in numerical_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Fill categorical missing values with mode
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print(f"✓ Missing values handled")
print(f"Remaining missing values: {df_clean.isnull().sum().sum()}")
print(f"Numerical features: {len(numerical_cols)}")
print(f"Categorical features: {len(categorical_cols)}")

In [ ]:
# Encode categorical features
df_encoded = df_clean.copy()

# Encode categorical features with LabelEncoder
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

if len(categorical_cols) > 0:
    print(f"✓ Categorical features encoded: {categorical_cols}")
else:
    print("✓ No categorical features to encode")

In [ ]:
# Encode target variable
# Check if target is already numeric (0/1) or needs encoding
if df_encoded[target_col].dtype == 'object':
    # Target is categorical, needs encoding
    le_target = LabelEncoder()
    y = le_target.fit_transform(df_encoded[target_col])
    print(f"Target encoding: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}")
else:
    # Target is already numeric
    y = df_encoded[target_col].values
    print(f"Target is already numeric: {np.unique(y)}")

print(f"Target shape: {y.shape}")
print(f"Class distribution: {np.bincount(y)}")

## 5. Create Feature Set

All models use the same features - ensemble diversity comes from different algorithms, not different data.

In [ ]:
# Remove target from features
X_all = df_encoded.drop(columns=[target_col])

print(f"All available features ({len(X_all.columns)}):")
print(X_all.columns.tolist())

In [ ]:
# Define features - all models use the same features
all_features = [
    'person_age',
    'person_income',
    'person_home_ownership',
    'person_emp_length',
    'loan_intent',
    'loan_grade',
    'loan_amnt',
    'loan_int_rate',
    'loan_percent_income',
    'cb_person_default_on_file',
    'cb_person_cred_hist_length'
]

print("Feature Set:")
print("="*60)
print(f"All models use ALL {len(all_features)} features:")
for f in all_features:
    print(f"  - {f}")
print("\nEnsemble diversity comes from different algorithms analyzing")
print("the same data in different ways:")
print("  - XGBoost: Gradient boosting with complex interactions")
print("  - LightGBM: Native categorical handling & leaf-wise growth")
print("  - Sklearn: Linear coefficients with L2 regularization")
print("="*60)

In [ ]:
# Validate features exist in dataset
available_features = X_all.columns.tolist()
all_features = [f for f in all_features if f in available_features]

print("✓ Feature set validated")
print(f"Features available: {len(all_features)}")
print(f"Total features in dataset: {len(available_features)}")
print(f"Coverage: {len(all_features)/len(available_features)*100:.1f}%")

In [ ]:
# Create feature matrices (all models use same data)
X_all_models = X_all[all_features]

print(f"Feature matrix shape: {X_all_models.shape}")
print(f"Features: {len(all_features)}")

## 6. Train/Test Split

In [ ]:
# Train/test split (same for all models)
X_train, X_test, y_train, y_test = train_test_split(
    X_all_models, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {len(y_train):,} samples")
print(f"Test set size: {len(y_test):,} samples")
print(f"\nFeature matrix shape:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test: {X_test.shape}")
print(f"\nClass distribution (train): {np.bincount(y_train)}")
print(f"Class distribution (test): {np.bincount(y_test)}")

## 7. Save Processed Data

In [ ]:
# Create processed data directory
os.makedirs('data/processed', exist_ok=True)

# Save single dataset (all models use the same data)
np.save('data/processed/X_train.npy', X_train.values)
np.save('data/processed/X_test.npy', X_test.values)
np.save('data/processed/y_train.npy', y_train)
np.save('data/processed/y_test.npy', y_test)

# Save feature names
with open('data/processed/feature_names.txt', 'w') as f:
    f.write('\n'.join(all_features))

print("✓ Data saved to data/processed/")
print("\nSaved files:")
print(f"  X_train.npy - shape {X_train.shape}")
print(f"  X_test.npy - shape {X_test.shape}")
print(f"  y_train.npy - {len(y_train):,} samples")
print(f"  y_test.npy - {len(y_test):,} samples")
print(f"  feature_names.txt - {len(all_features)} features")
print("\n✓ All models will use this same dataset")

## Summary

Data preparation complete!

**Feature Strategy**:

All three models receive **all 11 features**. Ensemble diversity comes from different algorithms analyzing the same data in different ways:

1. **XGBoost** → Gradient boosting with complex non-linear interactions
   - Captures intricate feature combinations
   - Tree-based splits find threshold effects

2. **LightGBM** → Native categorical handling & leaf-wise tree growth
   - Efficiently processes categorical features (loan_intent, home_ownership, etc.)
   - Faster training with histogram-based splitting

3. **Sklearn Logistic Regression** → Linear model with interpretable coefficients
   - Provides baseline linear relationships
   - Coefficient values explain feature importance for compliance

**Why This Works:**
- Different algorithms make different mistakes on the same data
- Combining predictions reduces overall error
- Each model contributes unique insights to the ensemble

**Next Steps**: Train each model on the full feature set:
1. `01_train_xgboost.ipynb`
2. `02_train_lightgbm.ipynb`
3. `03_train_sklearn.ipynb`

In [ ]:
# Final summary
print("="*60)
print("DATA PREPARATION SUMMARY")
print("="*60)
print(f"Total samples: {len(df_clean):,}")
print(f"Training samples: {len(y_train):,} ({len(y_train)/len(df_clean)*100:.1f}%)")
print(f"Test samples: {len(y_test):,} ({len(y_test)/len(df_clean)*100:.1f}%)")
print(f"\nAll models use the same {len(all_features)} features:")
for f in all_features:
    print(f"  - {f}")
print("="*60)
print("✓ Ready for ensemble training!")